In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SEED = 42
DATA_PATH = Path("Merged_Dataset_yoy.csv")

# Prediction targets: EBITDA, Net_Income, ROA (direction or value)
PREDICTION_TARGETS = ["EBITDA", "Net_Income", "ROA"]
TASKS = ["direction", "value"]  # "direction" for binary (up/down) and "value" for regression

DEFAULT_LOOKBACK_DAYS = 365  # Full year of daily data

# Extensive hyperparameter grid (36 configurations)
DEFAULT_EXPERIMENTS = [
    {"hidden_size": hs, "lr": lr, "batch_size": bs, "weight_decay": wd}
    for hs in [32, 64, 128]
    for lr in [1e-3, 3e-4, 1e-4]
    for bs in [16, 32]
    for wd in [1e-5, 1e-4]
]

MAX_EPOCHS = 100
PATIENCE = 10
TRAIN_YEAR_CUTOFF = 2019
VALID_YEAR_CUTOFF = 2021

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads() // 2))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"Configured Tasks: {TASKS}")
print(f"Total experiments per target: {len(DEFAULT_EXPERIMENTS)}")

Using device: cuda
Configured Tasks: ['direction', 'value']
Total experiments per target: 36


# Shallow LSTM benchmark

This notebook builds a single-layer LSTM for both a binary next-day direction task and a regression task on year-over-year value change.
The setup keeps the evaluation chronological, fits normalization only on training data, and reports both overfitting signals and temporal drift on the held-out period.

In [6]:
def load_and_prepare_yoy_data(
    data_path: Path, lookback_days: int = 365
) -> tuple[pd.DataFrame, list[str]]:
    """
    Load combined daily + targets data, build year-over-year sequences.
    Identifies year-end dates (where targets are not NaN) and extracts lookback_days
    of daily features ending at each year-end, then creates YoY direction and value labels.
    """
    df = pd.read_csv(data_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Company", "Date"]).reset_index(drop=True)
    df["year"] = df["Date"].dt.year

    # Identify year-end records (where at least one target is not NaN)
    df["has_targets"] = df[PREDICTION_TARGETS].notna().any(axis=1)
    year_end_records = df[df["has_targets"]].copy()

    # Get all numeric feature columns (excluding date/company/target metrics)
    exclude_cols = {"Date", "Company", "year", "has_targets"} | set(PREDICTION_TARGETS)
    feature_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

    # Build sequences
    sequences = []
    for company in df["Company"].unique():
        company_records = year_end_records[year_end_records["Company"] == company].copy()
        company_data = df[df["Company"] == company].copy()

        for _, year_end_row in company_records.iterrows():
            year = int(year_end_row["year"])
            year_end_date = year_end_row["Date"]

            # Get prior year record for comparison
            prior_year_records = year_end_records[
                (year_end_records["Company"] == company) & (year_end_records["year"] == year - 1)
            ]
            if prior_year_records.empty:
                continue

            prior_row = prior_year_records.iloc[0]

            # Extract window of daily data
            window_start = year_end_date - pd.Timedelta(days=lookback_days)
            window = company_data[
                (company_data["Date"] > window_start) & (company_data["Date"] <= year_end_date)
            ].copy()

            if len(window) < 200:  # Need sufficient data
                continue

            # Fill NaNs in features within window (bidirectional: forward then backward)
            window[feature_cols] = window[feature_cols].ffill().bfill()
            if window[feature_cols].isna().any().any():
                continue

            sequences.append(
                {
                    "company": company,
                    "year": year,
                    "year_end_date": year_end_date,
                    "window_data": window[feature_cols].to_numpy(dtype=np.float32),
                    "current_ebitda": year_end_row["EBITDA"],
                    "current_net_income": year_end_row["Net_Income"],
                    "current_roa": year_end_row["ROA"],
                    "prior_ebitda": prior_row["EBITDA"],
                    "prior_net_income": prior_row["Net_Income"],
                    "prior_roa": prior_row["ROA"],
                }
            )

    # Create targets for each prediction target
    data_records = []
    for seq in sequences:
        for target_name in PREDICTION_TARGETS:
            current_val = seq[f"current_{target_name.lower()}"]
            prior_val = seq[f"prior_{target_name.lower()}"]

            if pd.isna(current_val) or pd.isna(prior_val):
                continue

            label_direction = 1 if current_val > prior_val else 0
            label_value = (current_val - prior_val) / (np.abs(prior_val) + 1e-8)  # Normalized change

            data_records.append(
                {
                    "company": seq["company"],
                    "year": seq["year"],
                    "year_end_date": seq["year_end_date"],
                    "target": target_name,
                    "label_direction": label_direction,
                    "label_value": label_value,
                    "window_data": seq["window_data"],
                }
            )

    return pd.DataFrame(data_records), feature_cols


data_df, feature_cols = load_and_prepare_yoy_data(DATA_PATH, lookback_days=DEFAULT_LOOKBACK_DAYS)

print(f"Total sequences collected: {len(data_df):,}")
print(f"Features per sequence: {len(feature_cols)}")
print(f"Date range: {data_df['year_end_date'].min().date()} to {data_df['year_end_date'].max().date()}" if len(data_df) > 0 else "No data")
print(f"Companies: {data_df['company'].nunique()}" if len(data_df) > 0 else "No data")
print(f"\nBreakdown by target:")
for target in PREDICTION_TARGETS:
    target_data = data_df[data_df["target"] == target]
    if len(target_data) > 0:
        pos_rate = target_data["label_direction"].mean()
        print(f"  {target}: {len(target_data)} samples, positive rate: {pos_rate:.4f}")
    else:
        print(f"  {target}: 0 samples")

Total sequences collected: 3,247
Features per sequence: 25
Date range: 2010-12-30 to 2024-12-30
Companies: 93

Breakdown by target:
  EBITDA: 1068 samples, positive rate: 0.6320
  Net_Income: 1090 samples, positive rate: 0.5853
  ROA: 1089 samples, positive rate: 0.5142


In [7]:
class YoYSequenceDataset(Dataset):
    """Dataset for year-over-year fundamental prediction."""

    def __init__(self, data_df: pd.DataFrame, max_seq_length: int, task: str, scaler: StandardScaler = None):
        self.data_df = data_df.reset_index(drop=True)
        self.max_seq_length = max_seq_length
        self.task = task
        self.scaler = scaler

    def __len__(self) -> int:
        return len(self.data_df)

    def __getitem__(self, idx: int):
        row = self.data_df.iloc[idx]
        window = row["window_data"].copy()  # shape: (seq_len, features)

        if self.scaler is not None:
            window = self.scaler.transform(window)

        # Pad to max_seq_length (PRE-PADDING: add zeros at beginning so LSTM processes real data last)
        seq_len = len(window)
        if seq_len < self.max_seq_length:
            padding = np.zeros((self.max_seq_length - seq_len, window.shape[1]), dtype=np.float32)
            window = np.vstack([padding, window])

        label_col = f"label_{self.task}"
        target = np.float32(row[label_col])
        return torch.from_numpy(window), torch.tensor(target)


def split_by_year(data_df: pd.DataFrame, train_cutoff: int, valid_cutoff: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split data chronologically by year."""
    train_data = data_df[data_df["year"] <= train_cutoff].copy()
    val_data = data_df[(data_df["year"] > train_cutoff) & (data_df["year"] <= valid_cutoff)].copy()
    test_data = data_df[data_df["year"] > valid_cutoff].copy()
    return train_data, val_data, test_data

In [8]:

# Split data chronologically by year
train_data, val_data, test_data = split_by_year(data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

print(f"\nChronological split (train ≤ {TRAIN_YEAR_CUTOFF}, val ≤ {VALID_YEAR_CUTOFF}, test > {VALID_YEAR_CUTOFF}):")
print(f"  Train: {len(train_data)} samples ({train_data['year'].min():.0f}–{train_data['year'].max():.0f})")
print(f"  Val: {len(val_data)} samples ({val_data['year'].min():.0f}–{val_data['year'].max():.0f})")
print(f"  Test: {len(test_data)} samples ({test_data['year'].min():.0f}–{test_data['year'].max():.0f})")

# Fit StandardScaler only on training data to prevent leakage
scaler = StandardScaler()
train_windows = np.vstack([row for row in train_data["window_data"]])
scaler.fit(train_windows)

print(f"\nScaler fitted on {len(train_windows)} training windows")

# Compute max sequence length
max_seq_length = max(len(row) for row in data_df["window_data"])
print(f"Max sequence length: {max_seq_length}")


Chronological split (train ≤ 2019, val ≤ 2021, test > 2021):
  Train: 1948 samples (2010–2019)
  Val: 497 samples (2020–2021)
  Test: 802 samples (2022–2024)

Scaler fitted on 489489 training windows
Max sequence length: 256


In [9]:
class ShallowLSTMClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, task: str = "direction"):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.head = nn.Linear(hidden_size, 1)
        self.task = task

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)
        hidden = self.dropout(lstm_out[:, -1, :])
        logits = self.head(hidden).squeeze(-1)
        return logits


@torch.no_grad()
def collect_predictions(
    model: nn.Module, loader: DataLoader, criterion: nn.Module, task: str = "direction"
) -> tuple[dict, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    total_count = 0
    all_preds: list[np.ndarray] = []
    all_targets: list[np.ndarray] = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        batch_size = len(batch_y)
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size

        if task == "direction":
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
        else:
            all_preds.append(logits.cpu().numpy())

        all_targets.append(batch_y.cpu().numpy())

    if total_count == 0:
        empty_metrics = {"loss": np.nan, "accuracy": np.nan, "roc_auc": np.nan}
        return empty_metrics, np.array([]), np.array([])

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    if task == "direction":
        y_pred_binary = (y_pred >= 0.5).astype(int)
        try:
            auc = roc_auc_score(y_true.astype(int), y_pred)
        except ValueError:
            auc = np.nan
        metrics = {
            "loss": total_loss / total_count,
            "accuracy": accuracy_score(y_true.astype(int), y_pred_binary),
            "roc_auc": auc,
        }
        return metrics, y_true.astype(int), y_pred
    else:
        mae = np.mean(np.abs(y_pred - y_true))
        metrics = {
            "loss": total_loss / total_count,
            "mae": mae,
        }
        return metrics, y_true, y_pred


def train_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, task: str = "direction") -> dict:
    model.train()
    total_loss = 0.0
    total_count = 0
    all_preds: list[np.ndarray] = []
    all_targets: list[np.ndarray] = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = len(batch_y)
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size

        if task == "direction":
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_preds.append(probs)
        else:
            all_preds.append(logits.detach().cpu().numpy())

        all_targets.append(batch_y.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    if task == "direction":
        y_pred_binary = (y_pred >= 0.5).astype(int)
        try:
            auc = roc_auc_score(y_true.astype(int), y_pred)
        except ValueError:
            auc = np.nan
        return {
            "loss": total_loss / total_count,
            "accuracy": accuracy_score(y_true.astype(int), y_pred_binary),
            "roc_auc": auc,
        }
    else:
        mae = np.mean(np.abs(y_pred - y_true))
        return {"loss": total_loss / total_count, "mae": mae}


def make_loaders(train_data: pd.DataFrame, val_data: pd.DataFrame, test_data: pd.DataFrame, batch_size: int, max_seq_length: int, task: str):
    train_ds = YoYSequenceDataset(train_data, max_seq_length=max_seq_length, task=task, scaler=scaler)
    val_ds = YoYSequenceDataset(val_data, max_seq_length=max_seq_length, task=task, scaler=scaler)
    test_ds = YoYSequenceDataset(test_data, max_seq_length=max_seq_length, task=task, scaler=scaler)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

    return train_loader, val_loader, test_loader


def fit_single_experiment(target_name: str, config: dict, max_seq_length: int, task: str):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # Filter data for this target
    target_train = train_data[train_data["target"] == target_name].copy()
    target_val = val_data[val_data["target"] == target_name].copy()
    target_test = test_data[test_data["target"] == target_name].copy()

    if len(target_train) < 5 or len(target_val) < 2 or len(target_test) < 2:
        return None

    train_loader, val_loader, test_loader = make_loaders(target_train, target_val, target_test, config["batch_size"], max_seq_length, task)

    model = ShallowLSTMClassifier(input_size=len(feature_cols), hidden_size=config["hidden_size"], task=task).to(DEVICE)

    if task == "direction":
        pos_weight = torch.tensor(
            (len(target_train) - target_train["label_direction"].sum()) / max(1, target_train["label_direction"].sum()),
            dtype=torch.float32,
            device=DEVICE,
        )
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.HuberLoss(delta=1.0)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    history_rows = []
    best_state = None
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = train_epoch(model, train_loader, criterion, optimizer, task=task)
        val_metrics, _, _ = collect_predictions(model, val_loader, criterion, task=task)
        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "val_loss": val_metrics["loss"],
                **(
                    {"train_acc": train_metrics.get("accuracy"), "val_acc": val_metrics.get("accuracy")}
                    if task == "direction"
                    else {"train_mae": train_metrics.get("mae"), "val_mae": val_metrics.get("mae")}
                ),
            }
        )

        if val_metrics["loss"] < best_val_loss - 1e-4:
            best_val_loss = val_metrics["loss"]
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    train_metrics, _, _ = collect_predictions(model, train_loader, criterion, task=task)
    val_metrics, _, _ = collect_predictions(model, val_loader, criterion, task=task)
    test_metrics, test_true, test_pred = collect_predictions(model, test_loader, criterion, task=task)

    target_test_frame = target_test.reset_index(drop=True).copy()
    target_test_frame["true"] = test_true
    target_test_frame["pred"] = test_pred if task != "direction" else (test_pred >= 0.5).astype(int)

    return {
        "target": target_name,
        "config": config,
        "model": model,
        "history": pd.DataFrame(history_rows),
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "test_frame": target_test_frame,
        "train_size": len(target_train),
        "val_size": len(target_val),
        "test_size": len(target_test),
    }

In [10]:
run_results_direction = []

print(f"\n{'#'*90}")
print("### RUNNING TASK: DIRECTION")
print(f"{'#'*90}")

for target_name in PREDICTION_TARGETS:
    print(f"\n{'='*90}")
    print(f"Training shallow LSTM | Task: DIRECTION | Target: {target_name}")
    print(f"{'='*90}")

    for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
        print(
            f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
        )
        result = fit_single_experiment(
            target_name=target_name,
            config=config,
            max_seq_length=max_seq_length,
            task="direction",
        )

        if result is not None:
            run_results_direction.append(result)
            metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
            metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}"
            metrics_str += (
                f" | Train acc: {result['train_metrics']['accuracy']:.4f}"
                f" | Val acc: {result['val_metrics']['accuracy']:.4f}"
                f" | Test acc: {result['test_metrics']['accuracy']:.4f}"
            )
            print(metrics_str)
        else:
            print("Skipped: insufficient samples")


##########################################################################################
### RUNNING TASK: DIRECTION
##########################################################################################

Training shallow LSTM | Task: DIRECTION | Target: EBITDA

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.4720 | Val loss: 0.4813 | Train acc: 0.6106 | Val acc: 0.6380 | Test acc: 0.6090

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.4776 | Val loss: 0.4822 | Train acc: 0.6122 | Val acc: 0.7362 | Test acc: 0.6278

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.4730 | Val loss: 0.4799 | Train acc: 0.6201 | Val acc: 0.6319 | Test acc: 0.5489

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.4730 | Val loss: 0.4751 | Train acc: 0.6201 | Val acc: 0.7362 | Test acc: 0.5038

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 0.4684 | Val loss: 0.4833 | Train acc: 0.6186 | Val a

In [11]:
run_results_value = []

print(f"\n{'#'*90}")
print("### RUNNING TASK: VALUE")
print(f"{'#'*90}")

for target_name in PREDICTION_TARGETS:
    print(f"\n{'='*90}")
    print(f"Training shallow LSTM | Task: VALUE | Target: {target_name}")
    print(f"{'='*90}")

    for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
        print(
            f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
        )
        result = fit_single_experiment(
            target_name=target_name,
            config=config,
            max_seq_length=max_seq_length,
            task="value",
        )

        if result is not None:
            run_results_value.append(result)
            metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
            metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}"
            metrics_str += (
                f" | Train MAE: {result['train_metrics']['mae']:.4f}"
                f" | Val MAE: {result['val_metrics']['mae']:.4f}"
                f" | Test MAE: {result['test_metrics']['mae']:.4f}"
            )
            print(metrics_str)
        else:
            print("Skipped: insufficient samples")


##########################################################################################
### RUNNING TASK: VALUE
##########################################################################################

Training shallow LSTM | Task: VALUE | Target: EBITDA

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.7679 | Val loss: 0.8593 | Train MAE: 1.0197 | Val MAE: 1.1488 | Test MAE: 1.2560

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.7679 | Val loss: 0.8592 | Train MAE: 1.0197 | Val MAE: 1.1486 | Test MAE: 1.2559

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.7389 | Val loss: 0.8599 | Train MAE: 0.9859 | Val MAE: 1.1468 | Test MAE: 1.2431

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.7389 | Val loss: 0.8596 | Train MAE: 0.9859 | Val MAE: 1.1463 | Test MAE: 1.2430

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 0.7668 | Val loss: 0.8594 | Train MAE: 1.0167 | Val MAE: 1.14

In [12]:
def summarize_results(results: list[dict]) -> pd.DataFrame:
    summary_rows = []
    for r in results:
        task = r["model"].task
        row = {
            "Task": task,
            "Target": r["target"],
            "Hidden Size": r["config"]["hidden_size"],
            "LR": r["config"]["lr"],
        }
        if task == "direction":
            row["Test Metric"] = f"Acc: {r['test_metrics']['accuracy']:.4f}"
        else:
            row["Test Metric"] = f"MAE: {r['test_metrics']['mae']:.4f}"
        summary_rows.append(row)

    return pd.DataFrame(summary_rows)

summary_direction = summarize_results(run_results_direction)
summary_value = summarize_results(run_results_value)

print("\nDirection summary")
print(summary_direction.to_string(index=False) if not summary_direction.empty else "No results")
print("\nValue summary")
print(summary_value.to_string(index=False) if not summary_value.empty else "No results")


Direction summary
     Task     Target  Hidden Size     LR Test Metric
direction     EBITDA           32 0.0010 Acc: 0.6090
direction     EBITDA           32 0.0010 Acc: 0.6278
direction     EBITDA           32 0.0010 Acc: 0.5489
direction     EBITDA           32 0.0010 Acc: 0.5038
direction     EBITDA           32 0.0003 Acc: 0.5865
direction     EBITDA           32 0.0003 Acc: 0.5602
direction     EBITDA           32 0.0003 Acc: 0.4586
direction     EBITDA           32 0.0003 Acc: 0.4586
direction     EBITDA           32 0.0001 Acc: 0.4662
direction     EBITDA           32 0.0001 Acc: 0.4662
direction     EBITDA           32 0.0001 Acc: 0.4699
direction     EBITDA           32 0.0001 Acc: 0.4699
direction     EBITDA           64 0.0010 Acc: 0.4662
direction     EBITDA           64 0.0010 Acc: 0.4662
direction     EBITDA           64 0.0010 Acc: 0.5451
direction     EBITDA           64 0.0010 Acc: 0.5451
direction     EBITDA           64 0.0003 Acc: 0.6241
direction     EBITDA       